# Task 2 — Classificatore manuale: **Naïve Bayes**

**Corso:** Fondamenti e Applicazioni del Machine Learning (FML 2026)

Questo notebook definisce, implementa e valuta **manualmente** il classificatore
**Naïve Bayes** su `manuale.csv`, come richiesto dal Task 2 (uno dei due
classificatori del gruppo).

> **Stile di codice.** Vettoriale (pandas/numpy: `groupby`, `crosstab`, `value_counts`,
> `apply`, `prod`, `idxmax`), senza cicli `for` espliciti, come nei notebook del corso.

**Riferimenti:** Witten et al., *Data Mining* (4ª ed.), cap. 4 — Lezione 5.

## 0. Caricamento dei dati

In [1]:
import pandas as pd
import numpy as np

In [2]:
m = pd.read_csv("../data/processed/manuale.csv")
m

,age,campaign,job,marital,education,housing,loan,contact,poutcome,y
0,35,3,admin.,single,professional.course,yes,no,cellular,nonexistent,1
1,53,1,blue-collar,married,unknown,no,no,cellular,nonexistent,0
2,36,3,blue-collar,married,basic.9y,yes,no,cellular,failure,0
3,61,1,retired,married,basic.4y,no,no,telephone,nonexistent,1
4,47,3,admin.,married,university.degree,no,no,telephone,nonexistent,0
5,36,4,blue-collar,married,unknown,yes,no,cellular,nonexistent,0
6,30,4,blue-collar,married,basic.6y,yes,no,cellular,nonexistent,0
7,31,1,admin.,single,high.school,yes,no,cellular,nonexistent,1
8,51,5,technician,married,university.degree,yes,no,cellular,nonexistent,1
9,35,1,blue-collar,divorced,basic.9y,no,no,cellular,failure,1


In [3]:
nominali = ["job", "marital", "education", "housing", "loan", "contact", "poutcome"]
numerici = ["age", "campaign"]
print("Nominali:", nominali)
print("Numerici:", numerici)
print("\nDistribuzione classe:")
print(m["y"].value_counts())

Nominali: ['job', 'marital', 'education', 'housing', 'loan', 'contact', 'poutcome']
Numerici: ['age', 'campaign']

Distribuzione classe:
y
1    6
0    6
Name: count, dtype: int64


---
## 1. Come funziona Naïve Bayes (teoria, Lezione 5)

Naïve Bayes si basa sulla **regola di Bayes**. Per un'istanza con attributi
$x_1, \dots, x_n$:

$$ P(c \mid x_1,\dots,x_n) \propto P(c)\cdot \prod_{i=1}^{n} P(x_i \mid c) $$

- $P(c)$ = probabilità **a priori** della classe;
- $P(x_i \mid c)$ = **verosimiglianza** dell'attributo data la classe;
- il prodotto assume l'**indipendenza** degli attributi (da qui "naïve").

Si sceglie la classe col prodotto più alto (regola **MAP**).

**Due tipi di attributo:**
- **nominali** → frequenze, con **stimatore di Laplace** (conteggi inizializzati a 1):
  $$ P(x_i \mid c) = \frac{(\text{conteggio}) + 1}{N_c + v_i} $$
- **numerici** → **distribuzione gaussiana**:
  $$ P(x_i \mid c) = \frac{1}{\sqrt{2\pi}\,\sigma}\, e^{-\frac{(x_i-\mu)^2}{2\sigma^2}} $$

## 2. Adattamento ai dati — calcolo **a mano** su un'istanza

Prendiamo come test la **riga 0**, addestrando sulle altre 11.

In [4]:
test = m.iloc[0]
train = m.drop(0).reset_index(drop=True)
print("Istanza di test (riga 0):")
print(test)
print("\nClasse reale:", int(test["y"]))

Istanza di test (riga 0):
age                           35
campaign                       3
job                       admin.
marital                   single
education    professional.course
housing                      yes
loan                          no
contact                 cellular
poutcome             nonexistent
y                              1
Name: 0, dtype: object

Classe reale: 1


### 2.1 Probabilità a priori

In [5]:
n = len(train)
n1 = int((train["y"] == 1).sum()); n0 = int((train["y"] == 0).sum())
print(f"P(y=1) = {n1}/{n} = {n1/n:.4f}")
print(f"P(y=0) = {n0}/{n} = {n0/n:.4f}")

P(y=1) = 5/11 = 0.4545
P(y=0) = 6/11 = 0.5455


### 2.2 Verosimiglianze nominali (stimatore di Laplace)

Per ogni classe e attributo calcoliamo $(count+1)/(N_c+v)$. Stampiamo il dettaglio
con `apply` (niente `for`).

In [6]:
def likelihood_nom(attr, val, cls):
    sub = train[train["y"] == cls]
    v = train[attr].nunique()
    count = int((sub[attr] == val).sum())
    return (count + 1) / (len(sub) + v), count, len(sub), v

In [7]:
def _riga_nom(attr, cls):
    p, c, Nc, v = likelihood_nom(attr, test[attr], cls)
    print(f"  P({attr}={test[attr]}|y={cls}) = ({c}+1)/({Nc}+{v}) = {p:.4f}")

def _blocco_nom(cls):
    print(f"--- Classe y={cls} ---")
    pd.Series(nominali).apply(lambda a: _riga_nom(a, cls))
    print()

_ = pd.Series([1, 0]).apply(_blocco_nom)

--- Classe y=1 ---
  P(job=admin.|y=1) = (1+1)/(5+6) = 0.1818
  P(marital=single|y=1) = (2+1)/(5+3) = 0.3750
  P(education=professional.course|y=1) = (0+1)/(5+6) = 0.0909
  P(housing=yes|y=1) = (2+1)/(5+3) = 0.3750
  P(loan=no|y=1) = (4+1)/(5+3) = 0.6250
  P(contact=cellular|y=1) = (4+1)/(5+2) = 0.7143
  P(poutcome=nonexistent|y=1) = (4+1)/(5+2) = 0.7143

--- Classe y=0 ---
  P(job=admin.|y=0) = (1+1)/(6+6) = 0.1667
  P(marital=single|y=0) = (0+1)/(6+3) = 0.1111
  P(education=professional.course|y=0) = (0+1)/(6+6) = 0.0833
  P(housing=yes|y=0) = (4+1)/(6+3) = 0.5556
  P(loan=no|y=0) = (5+1)/(6+3) = 0.6667
  P(contact=cellular|y=0) = (4+1)/(6+2) = 0.6250
  P(poutcome=nonexistent|y=0) = (5+1)/(6+2) = 0.7500



### 2.3 Verosimiglianze numeriche (gaussiana)

In [8]:
def likelihood_num(attr, val, cls):
    sub = train[train["y"] == cls][attr]
    mu, sigma = sub.mean(), sub.std(ddof=1)
    p = (1 / (np.sqrt(2*np.pi) * sigma)) * np.exp(-((val - mu)**2) / (2 * sigma**2))
    return p, mu, sigma

In [9]:
def _riga_num(attr, cls):
    p, mu, sig = likelihood_num(attr, test[attr], cls)
    print(f"  P({attr}={test[attr]}|y={cls}): mu={mu:.2f}, sigma={sig:.2f} -> densita={p:.5f}")

def _blocco_num(cls):
    print(f"--- Classe y={cls} ---")
    pd.Series(numerici).apply(lambda a: _riga_num(a, cls))
    print()

_ = pd.Series([1, 0]).apply(_blocco_num)

--- Classe y=1 ---
  P(age=35|y=1): mu=39.80, sigma=16.04 -> densita=0.02379
  P(campaign=3|y=1): mu=1.80, sigma=1.79 -> densita=0.17808

--- Classe y=0 ---
  P(age=35|y=0): mu=40.83, sigma=8.42 -> densita=0.03726
  P(campaign=3|y=0): mu=2.83, sigma=1.17 -> densita=0.33780



### 2.4 Combinazione: posterior proporzionale

Moltiplichiamo prior × tutte le verosimiglianze. Implementazione vettoriale:
`crosstab` per i nominali, `groupby` per i numerici, `.prod()` per il prodotto.

In [10]:
def naive_bayes_score(istanza, train_df):
    prior = train_df["y"].value_counts(normalize=True)
    def lk_nom(a):
        tab = pd.crosstab(train_df[a], train_df["y"])
        v = train_df[a].nunique()
        Nc = train_df["y"].value_counts()
        return (tab.reindex([istanza[a]]).fillna(0).iloc[0] + 1) / (Nc + v)
    def lk_num(a):
        st = train_df.groupby("y")[a].agg(["mean", "std"])
        return (1/(np.sqrt(2*np.pi)*st["std"])) * np.exp(-((istanza[a]-st["mean"])**2)/(2*st["std"]**2))
    nom = pd.Series(nominali).apply(lk_nom)
    num = pd.Series(numerici).apply(lk_num)
    return prior * pd.concat([nom, num]).prod()

In [11]:
scores = naive_bayes_score(test, train)
print(f"score(y=0) = {scores[0]:.3e}")
print(f"score(y=1) = {scores[1]:.3e}")
pred = scores.idxmax()
print(f"\nPredizione: y={pred}  |  Classe reale: y={int(test['y'])}")
print("ESITO:", "corretto" if pred == test["y"] else "ERRATO")

score(y=0) = 1.839e-06
score(y=1) = 1.427e-06

Predizione: y=0  |  Classe reale: y=1
ESITO: ERRATO


> **Osservazione.** Su questa istanza Naïve Bayes predice `y=0`, mentre la classe
> reale è `y=1`: un errore. Con così pochi dati le stime di probabilità sono
> instabili e alcuni attributi nominali "tirano" verso la classe 0.

## 3. Implementazione completa — leave-one-out

In [12]:
pred_nb = pd.Series(m.index, index=m.index).apply(
    lambda i: naive_bayes_score(m.iloc[i], m.drop(i).reset_index(drop=True)).idxmax())
acc_nb = (pred_nb == m["y"]).mean()
print(f"Accuratezza Naive Bayes (leave-one-out): {acc_nb:.2%}")

Accuratezza Naive Bayes (leave-one-out): 50.00%


## 4. Confronto con l'API di Scikit-Learn

La traccia consente l'uso di API. Usiamo `GaussianNB` come controprova (sklearn
tratta i nominali come numeri, quindi prima li codifichiamo: il risultato non sarà
identico al nostro, che distingue nominali e numerici). Usiamo `cross_val_predict`
con `LeaveOneOut` per evitare cicli espliciti.

In [13]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import LeaveOneOut, cross_val_predict

X = m[numerici + nominali].copy()
X[nominali] = OrdinalEncoder().fit_transform(X[nominali])
y = m["y"]

preds = cross_val_predict(GaussianNB(), X, y, cv=LeaveOneOut())
print(f"Accuratezza GaussianNB (sklearn, LOO): {(preds == y).mean():.2%}")

Accuratezza GaussianNB (sklearn, LOO): 58.33%


---
## Riepilogo

- **Naïve Bayes** combina **tutti** gli attributi (nominali con Laplace, numerici con
  gaussiana) tramite la regola di Bayes.
- Assume **indipendenza** degli attributi e **normalità** dei numerici: assunzioni
  forti, soprattutto su pochi dati.
- L'accuratezza in **leave-one-out** è un riferimento da confrontare con 1R (vedi
  notebook dedicato): Naïve Bayes sfrutta più informazione di 1R.

Le metriche su 12 istanze sono poco affidabili: illustrano il funzionamento, non
giudicano il modello (valutazione seria nei Task 4–5).